In [6]:
!pip install ccxt

In [9]:
import numpy as np

def calculate_sharpe_ratio(returns, risk_free_rate=0.0459, periods_per_year=252):
    returns = np.array(returns)
    excess_returns = returns - risk_free_rate / periods_per_year
    annualized_excess = np.mean(excess_returns) * periods_per_year
    annualized_vol = np.std(returns, ddof=1) * np.sqrt(periods_per_year)
    sharpe = annualized_excess / annualized_vol if annualized_vol > 0 else 0.0
    return sharpe

def calculate_max_drawdown(equity_curve):
    equity = np.array(equity_curve)
    running_max = np.maximum.accumulate(equity)
    drawdowns = (running_max - equity) / running_max
    max_dd_idx = np.argmax(drawdowns)
    max_dd = drawdowns[max_dd_idx]
    peak_idx = np.argmax(equity[:max_dd_idx + 1]) if max_dd_idx > 0 else 0
    return max_dd, peak_idx, max_dd_idx

def backtest_summary(returns, equity_curve, risk_free_rate=0.0459):
    sharpe = calculate_sharpe_ratio(returns, risk_free_rate)
    max_dd, peak_idx, trough_idx = calculate_max_drawdown(equity_curve)
    annual_return = np.mean(returns) * 252
    calmar = annual_return / max_dd if max_dd > 0 else np.inf
    return {
        'sharpe_ratio': round(sharpe, 3),
        'max_drawdown': round(max_dd, 4),
        'max_drawdown_pct': f"{max_dd * 100:.2f}%",
        'calmar_ratio': round(calmar, 3),
        'annual_return': f"{annual_return * 100:.2f}%",
        'annualized_vol': f"{np.std(returns, ddof=1) * np.sqrt(252) * 100:.2f}%",
    }
    np.random.seed(42)

n_days = 252
high_sharpe = np.random.normal(0.001, 0.008, n_days)
high_sharpe_equity = 100 * np.cumprod(1 + high_sharpe)

low_sharpe = np.random.normal(0.0008, 0.025, n_days)
low_sharpe_equity = 100 * np.cumprod(1 + low_sharpe)

trap_sharpe = np.random.normal(0.0015, 0.006, n_days)
trap_sharpe[200] = -0.25
trap_sharpe_equity = 100 * np.cumprod(1 + trap_sharpe)

print("=== 策略A：稳定型（高夏普）===")
for k, v in backtest_summary(high_sharpe, high_sharpe_equity).items():
    print(f"  {k}: {v}")

print("\n=== 策略B：波动型（低夏普）===")
for k, v in backtest_summary(low_sharpe, low_sharpe_equity).items():
    print(f"  {k}: {v}")

print("\n=== 策略C：高夏普陷阱（一次崩盘带走）===")
for k, v in backtest_summary(trap_sharpe, trap_sharpe_equity).items():
    print(f"  {k}: {v}")

=== 策略A：稳定型（高夏普）===
  sharpe_ratio: 1.125
  max_drawdown: 0.0628
  max_drawdown_pct: 6.28%
  calmar_ratio: 3.013
  annual_return: 18.93%
  annualized_vol: 12.75%

=== 策略B：波动型（低夏普）===
  sharpe_ratio: -0.239
  max_drawdown: 0.3102
  max_drawdown_pct: 31.02%
  calmar_ratio: -0.161
  annual_return: -4.99%
  annualized_vol: 40.02%

=== 策略C：高夏普陷阱（一次崩盘带走）===
  sharpe_ratio: 0.304
  max_drawdown: 0.2506
  max_drawdown_pct: 25.06%
  calmar_ratio: 0.509
  annual_return: 12.76%
  annualized_vol: 26.91%
